# Phase C — Komplexe Regression (Stufe 1)

Ein **2D-U-Net** verfeinert das Copy-Up-gefüllte Input-Spektrogramm zum echten HF
(RI+Mag-Loss, nur HF). Es ist die *verwaschene* Hälfte der Story und liefert die
Warm-Start-Gewichte fürs GAN. Importiert aus `bwe/` — nicht self-contained.

Training läuft auf Kaggle (`kaggle_train_regression.ipynb`); die Ergebnis-Zellen unten
greifen, sobald der unten gesetzte `REG_RUN` unter `BWE_CKPT_ROOT` trainiert ist (explizit
gesetzt — kein Wildcard, da unter `CKPT_ROOT` mehrere Läufe liegen).

In [ ]:
import os
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Audio, display
import tensorflow as tf

from bwe import config as cfg
from bwe.models.generator import Generator, build_unet
from bwe.data.loaders import load_demo
from bwe.infer.reconstruct import model_from_fullband, baseline_from_fullband
from bwe.eval import metrics as M
from bwe.eval import plots as P

# >>> Expliziter Run-Name (statt Wildcard — unter CKPT_ROOT liegen mehrere Läufe). <<<
REG_RUN = "reg_full_3"   # finaler Regressions-Lauf (latest-greatest)

print(cfg.summary())
print(f"REG_RUN={REG_RUN}")

## Architektur

Voll-faltend (4 Ebenen 32→64→128→256, Skip-Connections), Eingang 3 Kanäle (Re/Im/Freq-Koord), Ausgang 2 (Re/Im, linear). Stride-2 → F und T werden um 16 geteilt; die Zeit wird dynamisch gepaddet/gecroppt.

In [2]:
build_unet().summary()

Model: "bwe_unet"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_spec          │ (None, 512, None, │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ down0 (Sequential)  │ (None, 256, None, │      1,664 │ input_spec[0][0]  │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ down1 (Sequential)  │ (None, 128, None, │     33,024 │ down0[0][0]       │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ down2 (Sequential)  │ (None, 64, None,  │    131,584 │ down1[0][0]       │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ down3 (Sequential)  │ (None, 32, None,  │    525,312 │ down2[0][0]       │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ up0 (Sequential)    │ (None, 64, None,  │    524,800 │ down3[0][0]       │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 64, None,  │          0 │ up0[0][0],        │
│ (Concatenate)       │ 256)              │            │ down2[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ up1 (Sequential)    │ (None, 128, None, │    262,400 │ concatenate[0][0] │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 128, None, │          0 │ up1[0][0],        │
│ (Concatenate)       │ 128)              │            │ down1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ up2 (Sequential)    │ (None, 256, None, │     65,664 │ concatenate_1[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_2       │ (None, 256, None, │          0 │ up2[0][0],        │
│ (Concatenate)       │ 64)               │            │ down0[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ out                 │ (None, 512, None, │      2,050 │ concatenate_2[0]… │
│ (Conv2DTranspose)   │ 2)                │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,546,498 (5.90 MB)

 Trainable params: 1,545,090 (5.89 MB)

 Non-trainable params: 1,408 (5.50 KB)

## Trainingsprinzip

Pro Step: `Generator → Splicing (LF original, HF Modell) → RI+Mag-Loss (nur HF)`.
Validation ist **deterministisch** (feste Segmente, keine Augmentation) und liefert
**Val-LSD-HF** als echtes Qualitätsmaß für EarlyStopping/Checkpoint-Wahl.
Korrektheitsbeweis lokal: *einen* Batch auswendig lernen → Loss → ~0.

## Lernkurve (nach dem Training)

In [ ]:
log_path = cfg.CKPT_ROOT / REG_RUN / "log.csv"
if log_path.exists():
    df = pd.read_csv(log_path)
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(df["epoch"], df["loss"], label="train")
    if "val_loss" in df: ax[0].plot(df["epoch"], df["val_loss"], label="val")
    ax[0].set_title("RI+Mag-Loss"); ax[0].set_xlabel("Epoche"); ax[0].legend()
    if "val_lsd_hf" in df:
        ax[1].plot(df["epoch"], df["val_lsd_hf"]); ax[1].set_title("Val-LSD-HF [dB]")
        ax[1].set_xlabel("Epoche")
    plt.tight_layout(); plt.show()
else:
    print(f"Noch keine log.csv unter {log_path} — zuerst auf Kaggle trainieren ({REG_RUN}).")

## Regression vs. Copy-Up vs. Original (nach dem Training)

In [ ]:
w_path = cfg.CKPT_ROOT / REG_RUN / "generator.weights.h5"
if not w_path.exists():
    print(f"Noch kein trainiertes Modell unter {w_path} — zuerst auf Kaggle trainieren ({REG_RUN}).")
else:
    g = Generator()
    g(tf.zeros([1, cfg.N_BINS_NET, cfg.SEG_FRAMES, cfg.N_INPUT_CHANNELS]))  # bauen
    g.load_weights(str(w_path))
    print("Gewichte:", w_path)

    name, target = load_demo("valid", 0, seconds=6.0, offset=30.0)
    reg, inp = model_from_fullband(g, target)
    cu, _ = baseline_from_fullband(target)

    P.spectro_triple(inp, reg, target,
                     titles=("Bandbegrenzt (Input)", "Regression", "Original (Target)"))
    plt.show()

    print(f"{'':12s}{'LSD-HF [dB]':>14s}{'SI-SDR [dB]':>14s}")
    for lbl, w in [("Copy-Up", cu), ("Regression", reg)]:
        print(f"{lbl:12s}{M.lsd_hf(w, target):14.2f}{M.si_sdr(w, target):14.2f}")

    print("\nAnhören:");
    print("Bandbegrenzt:"); display(Audio(inp.numpy(), rate=cfg.SR))
    print("Regression:");   display(Audio(reg.numpy(), rate=cfg.SR))
    print("Original:");     display(Audio(target, rate=cfg.SR))

## Fazit

Die Regression senkt die **LSD-HF** unter die Copy-Up-Baseline, das HF wird *plausibler*
— bleibt aber **verwaschen** (Mittelungs-Loss auf einem ill-posed-Problem). Genau dieser
Mangel motiviert das GAN (Phase D): von „verwaschen" zu „scharf". Die Generator-Gewichte
dienen dort als Warm-Start.